# Review Radar — the live demo

One kernel, ten moves, 4:40 of a five-minute budget (ADR-0009, RR-11).

Every cell calls a function in `src/serving/demo.py`; no logic lives here. The moves are the
running order the rehearsal counts, and the cell below prints it — including the moves whose
phase is not built yet, which are **placed and marked**, never dropped.

**Before running anything**, in a real terminal, not in this notebook:

```
make up-ui          # the stack, plus Kibana for move 5
make health         # move 1 — four services proved usable from Python
```

Run this notebook top to bottom. It is committed with outputs stripped; an executed export
under `docs/demo/<date>/` is a rehearsal record (`DEMO_GATE`, ticket 18).

In [ ]:
from IPython.display import display

from src.serving import demo

display(demo.run_sheet())
demo.budget()

---
## Move 1 — the stack is real · terminal · 10 s

Said out loud: *four services proved usable from Python, in a real shell, before the notebook
opens.*

`make health` runs in the terminal, not here: a `!` cell proves the kernel can shell out, and
what is being claimed is that Kafka, MinIO, Elasticsearch and Postgres answer. The notebook's
own three connections open next, and they stay open for the whole demo.

In [ ]:
stage = demo.open_stage(scope="full")
demo.context(stage)

---
## Move 2 — the paced replay and the streaming projection · notebook · 20 s

Said out loud: *the file goes onto its own topic in event order at a constant rate; the
event-time clock moves while the rest of the demo runs.*

The replay is a background process (`src/ingest/stream_producer.py`, its own ledger run) and
the projection is this kernel's Spark session running the aggregation job's own functions.
Both keep going through moves 3–9; move 10 reads them.

**Placed and marked:** the injected near/far slices and the demo run's own `STREAM_GATE` are
the P8 demo run (ADR-0010, ticket 16). Until that exists this replays without injection, and
move 10 prints the **control** run's recorded gate.

`start_replay(reset=True)` deletes and refills the stream topic. That is what the demo wants
and what a development machine does not — it discards the topic the control run's gate
re-reads.

In [ ]:
live = demo.start_live_projection(stage)
live.replay = demo.start_replay(scope=stage.scope, reset=True)
print("\n".join(demo.replay_clock()))

---
## Move 3 — silver's reconciliation identity · notebook · 15 s

Said out loud: *every bronze row is accounted for: rejected, survived, or removed as a
duplicate.*

These are the lines `make gate-silver` printed, read back from `eval/silver/gate.json`. The
gate is a re-derivation over pinned snapshots and costs minutes; showing its artefact is the
same number without retyping it.

In [ ]:
demo.show_gate("silver")

---
## Move 4 — every run this data went through · notebook · 30 s

Said out loud: *the silver row names the bronze snapshot it read, and time travel reads that
snapshot back to the same count.*

Provenance as a join, not an assertion (ADR-0008, RR-16): the ledger row, then a Spark SQL
`VERSION AS OF` read of exactly the snapshot that row pinned.

In [ ]:
display(demo.ledger(stage, last=12))
demo.time_travel(stage)

---
## Move 5 — decline candidates · Kibana · 45 s

Said out loud: *the opening question, answered over the serving projection.*

The one move that leaves the notebook. The dashboard reads the `product_month` alias and
nothing else; individual reviews are shown from here, never from Kibana.

In [ ]:
print(demo.kibana_url())

---
## Move 6 — hybrid decomposition · notebook · 35 s

Said out loud: *a review neither list ranks first wins the fusion.*

One descriptive query from the frozen twenty, run through the production hybrid — the same
function P5 was evaluated on — with the per-leg ranks it already carries unpacked beside the
fused score `Σ 1/(60 + rank)`.

In [ ]:
table = demo.hybrid_decomposition(stage, demo.DEMO_QUERY)
display(table)
print(demo.DEMO_QUERY, "→", demo.fusion_story(table))

---
## Move 7 — cached complaint-theme labels · notebook · 30 s

Said out loud: *themes are hypotheses about what the text says, never a diagnosed cause, and
P6's published macro-F1 is a reported miss.*

Every row was written by `make label-themes` and is read back from `gold.review_theme_labels`:
the demo makes no model call. The counts are over this product's **labelled** reviews — the
seeded sample frames, not its whole history — so no share, direction or cause is claimed here.
The published P6 number is move 8's.

In [ ]:
candidate = demo.top_candidate()
print(demo.candidate_headline(candidate))
demo.theme_evidence(stage, candidate=candidate)

---
## Move 8 — the evaluation table · notebook · 20 s

Said out loud: *one row per capability, every bar set before the number beside it existed.*

`make eval-table`, rendered in this kernel. Reproducibility rows block their phase; quality
rows report beside their bars and never block. A capability with no artefact and no written
cut reason makes this table `INCOMPLETE` — silence is not one of the verdicts.

In [ ]:
print(demo.evaluation_table())

---
## Move 9 — RAG answers, claim by claim · notebook · 30 s

Said out loud: *every claim cites a review inside the window the question declared; it
contrasts cited examples and never says a complaint increased.*

The thirty were frozen before any answer existed and answered once. The table below resolves
each citation against that question's own retrieved set and checks the stored `review_month`
against the declared window — the two mechanical rules `RAG_GATE` blocks on, re-run live.

In [ ]:
display(demo.rag_questions())
qid = demo.clean_question(family="temporal")
print(qid, "—", demo.question_text(qid))
display(demo.citations(qid))
print(demo.citation_contract(qid))

---
## Move 10 — the stream's own gate · notebook · 25 s

Said out loud: *the projection is reconciled against batch product-month by product-month.*

The live query started at move 2 is stopped and says what it saw; then the recorded
`STREAM_GATE` — the control run, re-derived by `make gate-stream` from the topic, the sort
job's ledger row, silver's validation digest and the pinned gold snapshot.

**Placed and marked:** the alerts table and the demo run's watermark drops come with the P8
demo run (ticket 16).

In [ ]:
display(demo.stop_live_projection(live))
demo.show_gate("stream_control")

---
## After the clock stops

The exactly-once proof is not a live move (65 s of 300). It is a figure in the design doc and
a terminal transcript in the recorded backup, played only if asked:
`make eos`. A live `llama3.2:3b` label is a Q&A reserve, never on the clock.

In [ ]:
demo.close_stage(stage)